# Explore: activations, gradients & BatchNorm

This notebook is the **diagnostic dashboard** for your project - the visual side of every
milestone. It *consumes* the code you write in `mlp.py`, `nn.py`, and `train.py`; no
solutions live here. If a cell fails, it usually means its milestone isn't done yet -
`python check.py` is the source of truth.

Each section says which milestone it needs. Run cells top to bottom.

*Needs:* `pip install jupyter matplotlib` (on top of torch).

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from data import load_words, build_vocab, build_dataset, split_dataset
import baseline

words = load_words()
stoi, itos = build_vocab(words)
g = torch.Generator().manual_seed(42)
train_words, dev_words, test_words = split_dataset(words, generator=g)
Xtr, Ytr = build_dataset(train_words, stoi)
Xdev, Ydev = build_dataset(dev_words, stoi)
Xb, Yb = Xtr[:1024], Ytr[:1024]   # a working batch for the experiments below
print(f"train {Xtr.shape[0]:,} examples, dev {Xdev.shape[0]:,}")

## 1. The autopsy: why did Module 3 start at loss 27?  *(needs Milestone 1)*

A network that has seen nothing should have no opinion: every character equally likely,
loss = -log(1/27) = **3.3**. Your Module 3 network (preserved in `baseline.py`) scored
~27. Let's see the crime and your fix side by side.

In [ ]:
from mlp import init_params

naive = baseline.init_params_naive(generator=torch.Generator().manual_seed(2147483647))
fixed = init_params(generator=torch.Generator().manual_seed(2147483647))

with torch.no_grad():
    logits_naive = baseline.forward(Xb, naive)
    logits_fixed = baseline.forward(Xb, fixed)
    loss_naive = F.cross_entropy(logits_naive, Yb).item()
    loss_fixed = F.cross_entropy(logits_fixed, Yb).item()

print(f"uniform 'no idea' loss:  {torch.tensor(27.).log().item():.4f}")
print(f"naive init (Module 3):   {loss_naive:.4f}")
print(f"your fixed init:         {loss_fixed:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
axes[0].hist(logits_naive.detach().view(-1), bins=60, color="tomato")
axes[0].set_title(f"logits at init - naive (loss {loss_naive:.1f})")
axes[1].hist(logits_fixed.detach().view(-1), bins=60, color="seagreen")
axes[1].set_title(f"logits at init - yours (loss {loss_fixed:.2f})")
for ax in axes:
    ax.set_xlabel("logit value")
plt.show()

Same architecture, same data. The naive logits sprawl across tens of units - after
softmax that's near-certainty about *random garbage*, and cross-entropy punishes
confidently-wrong hardest. Your logits huddle near zero: maximum uncertainty, loss 3.3,
nothing to un-learn. Module 3 spent its first thousands of steps just undoing that sprawl
- the "hockey stick" at the start of its loss curve was pure waste.

## 2. The dead-neuron map  *(needs Milestone 2)*

The second problem hid in the hidden layer: with W1 at scale 1.0, `tanh` gets huge inputs
and pins to +/-1. Flat regions pass (chain rule) almost no gradient - a neuron that's
saturated for **every** example is dead: it will never learn. White pixels below are
saturated activations.

In [ ]:
from mlp import saturation

def hidden(X, params):
    C, W1, b1, W2, b2 = params
    emb = C[X].view(X.shape[0], -1)
    return torch.tanh(emb @ W1 + b1)

h_naive = hidden(Xb, naive).detach()
h_fixed = hidden(Xb, fixed).detach()
print(f"saturation (|h| > 0.97) - naive: {saturation(h_naive) * 100:5.1f}%")
print(f"saturation (|h| > 0.97) - yours: {saturation(h_fixed) * 100:5.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
axes[0].imshow(h_naive[:32].abs() > 0.99, cmap="gray", interpolation="nearest")
axes[0].set_title("naive init: 32 examples x 200 neurons (white = saturated)")
axes[1].imshow(h_fixed[:32].abs() > 0.99, cmap="gray", interpolation="nearest")
axes[1].set_title("your init")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(h_naive.view(-1), bins=50, color="tomato")
axes[0].set_title("tanh outputs - naive: piled on the rails")
axes[1].hist(h_fixed.view(-1), bins=50, color="seagreen")
axes[1].set_title("tanh outputs - yours: breathing room")
plt.show()

Look for an all-white **column** in the top map: that would be a fully dead neuron -
dead for every example, unreachable by gradient descent forever. With your fix the map
goes mostly black and the histogram spreads out. (In Milestone 3 you replace the eyeballed
scale with `kaiming_scale` - same picture, no magic number.)

## 3. BatchNorm tames a wild layer  *(needs Milestone 4)*

Careful init gets the scales right *at step 0*. BatchNorm **forces** them right at every
step, no matter what the weights get up to. Feed it the naive network's wild
preactivations and watch:

In [ ]:
from mlp import batchnorm, init_bn_params

C, W1, b1, W2, b2 = naive
hpreact_wild = (C[Xb].view(Xb.shape[0], -1) @ W1 + b1).detach()
gain, bias = init_bn_params(hidden=200)
hpreact_tamed = batchnorm(hpreact_wild, gain, bias).detach()

print(f"before: per-column std ranges "
      f"{hpreact_wild.std(0).min():.2f} .. {hpreact_wild.std(0).max():.2f}")
print(f"after:  per-column std ranges "
      f"{hpreact_tamed.std(0).min():.2f} .. {hpreact_tamed.std(0).max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(hpreact_wild.view(-1), bins=60, color="tomato")
axes[0].set_title("preactivations - wild (std ~5.5)")
axes[1].hist(hpreact_tamed.view(-1), bins=60, color="seagreen")
axes[1].set_title("after your batchnorm - unit gaussian")
plt.show()

The gain starts at 1 and the bias at 0, so BatchNorm begins as "make it standard
gaussian" - but both are **trainable**, so the network can undo the normalization exactly
where it profits from doing so. Normalized is the starting point, not a straitjacket.

## 4. Running stats: BatchNorm without a batch  *(needs Milestone 5)*

Batch statistics need a batch - but inference often has exactly **one** example. During
training you keep an exponential moving average of the mean/std; at inference you use it.
Watch the running estimate find the true value without ever seeing the full dataset:

In [ ]:
from mlp import update_running, batchnorm_infer

with torch.no_grad():
    hpre_all = C[Xtr].view(Xtr.shape[0], -1) @ W1 + b1
true_mean = hpre_all.mean(0, keepdim=True)
true_std = hpre_all.std(0, keepdim=True)

running_mean = torch.zeros((1, 200))
running_std = torch.ones((1, 200))
trace = []
gb = torch.Generator().manual_seed(7)
for i in range(600):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=gb)
    hb = hpre_all[ix]
    running_mean = update_running(running_mean, hb.mean(0, keepdim=True), momentum=0.01)
    running_std = update_running(running_std, hb.std(0, keepdim=True), momentum=0.01)
    trace.append(running_mean[0, 0].item())

plt.figure(figsize=(10, 3))
plt.plot(trace, label="running estimate (neuron 0 mean)")
plt.axhline(true_mean[0, 0].item(), color="k", linestyle="--",
            label="true mean over all 182k examples")
plt.xlabel("batches seen")
plt.legend()
plt.show()

In [ ]:
one = Xtr[:1]  # a batch of ONE - no batch statistics exist here
with torch.no_grad():
    hp = C[one].view(1, -1) @ W1 + b1
out = batchnorm_infer(hp, gain, bias, running_mean, running_std)
print("single-example inference:", "works, all finite"
      if torch.isfinite(out).all() else "BROKEN")

This is why `model.train()` / `model.eval()` exist in every real framework - and why
forgetting `.eval()` before inference is one of the most-Googled PyTorch bugs. You now
know exactly what breaks and why.

## 5. Your deep network, at birth  *(needs Milestone 6)*

Six linear layers, BatchNorm after each, 47,024 parameters. The lecture's signature
diagnostic: overlay the activation distribution of every Tanh layer. Healthy = the layers
look like *each other* - the signal neither explodes nor fades as it goes deeper.

In [ ]:
from nn import Linear, BatchNorm1d, Tanh, build_network, forward_net

C6, layers, params = build_network(generator=torch.Generator().manual_seed(2147483647))
print(f"{sum(p.nelement() for p in params):,} parameters")
with torch.no_grad():
    logits = forward_net(Xb, C6, layers)
    print(f"initial loss: {F.cross_entropy(logits, Yb).item():.3f}")

plt.figure(figsize=(12, 3.5))
for i, layer in enumerate(layers):
    if isinstance(layer, Tanh):
        t = layer.out.detach()
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1], hy, label=f"layer {i}")
plt.legend()
plt.title("activation distribution per Tanh layer - WITH BatchNorm")
plt.show()

In [ ]:
# the same tower without BatchNorm (and no compensating gain): watch the signal fade
gnb = torch.Generator().manual_seed(2147483647)
layers_nobn = [Linear(30, 100, generator=gnb), Tanh()]
for _ in range(4):
    layers_nobn += [Linear(100, 100, generator=gnb), Tanh()]
layers_nobn += [Linear(100, 27, generator=gnb)]

with torch.no_grad():
    forward_net(Xb, C6, layers_nobn)

plt.figure(figsize=(12, 3.5))
for i, layer in enumerate(layers_nobn):
    if isinstance(layer, Tanh):
        t = layer.out.detach()
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1], hy, label=f"layer {i} (std {t.std():.2f})")
plt.legend()
plt.title("activation distribution per Tanh layer - NO BatchNorm, gain 1")
plt.show()

Without BatchNorm, `1/sqrt(fan_in)` alone isn't enough: every tanh squashes the
signal a little, so each layer's distribution is narrower than the last - stack enough of
them and the network goes silent (that's what tanh's 5/3 gain is for). With BatchNorm the
layers are indistinguishable, *by construction*. That robustness is what BatchNorm buys:
this shallow network barely improves its final loss, but a very deep one becomes trainable
at all.

## 6. Training telemetry  *(needs Milestone 7)*

Now train for real - 1,000 steps - while your instrumentation records everything.
Professionals don't stare at the loss alone; these are the plots they watch instead.

In [ ]:
from train import activation_stats, train, update_ratios

C6, layers, params = build_network(generator=torch.Generator().manual_seed(2147483647))
lossi, ud = train(Xtr, Ytr, C6, layers, params, steps=1000, lr=0.1,
                  generator=torch.Generator().manual_seed(2147483647))

plt.figure(figsize=(10, 3))
plt.plot(torch.tensor(lossi).log10())
plt.xlabel("step")
plt.ylabel("log10 loss")
plt.title(f"loss: {lossi[0]:.2f} -> ~{sum(lossi[-50:]) / 50:.2f} "
          "(no hockey stick - nothing to un-learn)")
plt.show()

In [ ]:
# the health report, after training
for s in activation_stats(layers):
    print(f"layer {s['layer']:2d} (Tanh): mean {s['mean']:+.2f}  "
          f"std {s['std']:.2f}  saturated {s['saturated'] * 100:4.1f}%")

plt.figure(figsize=(12, 3.5))
for i, layer in enumerate(layers):
    if isinstance(layer, Tanh):
        t = layer.out.detach()
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1], hy, label=f"layer {i}")
plt.legend()
plt.title("activations after 1,000 steps - still healthy")
plt.show()

In [ ]:
# gradients flowing through every Tanh (retain_grad in your train() makes this possible)
plt.figure(figsize=(12, 3.5))
for i, layer in enumerate(layers):
    if isinstance(layer, Tanh):
        t = layer.out.grad
        hy, hx = torch.histogram(t, density=True)
        plt.plot(hx[:-1], hy, label=f"layer {i} (std {t.std():.1e})")
plt.legend()
plt.title("gradient distribution per Tanh layer - similar scale everywhere")
plt.show()

In [ ]:
# the update:data ratio - how big is each step relative to the weights it moves?
plt.figure(figsize=(10, 3.5))
n_2d = len([p for p in params if p.ndim == 2])
for i in range(n_2d):
    plt.plot([ud[j][i] for j in range(len(ud))])
plt.axhline(-3, color="k", linestyle="--", label="the -3 line (updates ~1/1000 of data)")
plt.xlabel("step")
plt.ylabel("log10(update / data)")
plt.legend()
plt.show()

The rule of thumb: healthy update:data ratios hover around **-3** (each step moves a
weight matrix by about a thousandth of its size). Well below -3 for many layers means the
learning rate is too timid; well above means it's reckless. Try it: retrain the cell above
with `lr=0.01` and watch every line sink by exactly 1.

## 7. The name playground  *(needs Milestone 7)*

Sampling happens one character at a time - a batch of ONE - which only works because your
BatchNorm layers can fall back on their running statistics. Flip them to inference mode
and generate:

In [ ]:
from train import evaluate, sample_name

for layer in layers:
    layer.training = False   # the line every PyTorch bug report forgets

print(f"val loss after 1,000 steps: {evaluate(Xdev, Ydev, C6, layers):.4f} "
      "(Module 3 needed 40,000 steps to reach 2.2564)")
gs = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(10):
    print(" ", sample_name(C6, layers, itos, generator=gs))

Change the seed for different names; retrain longer in section 6 for better ones.

## The full payoff

`python train.py` runs the real schedule - 12,000 steps - and ends **past your Module 3
number** with a first loss of ~3.3. That's the whole module in one line of output: same
task, same data, but a network that starts sane, trains healthy, and can prove it.

Next lecture: [makemore Part 4 - Becoming a Backprop Ninja](https://www.youtube.com/watch?v=q8SA3rM6ckI)
- `loss.backward()` gets rebuilt by hand through every layer you just stacked, BatchNorm
included.